# BUSI70575 Final Reproducible Pipeline

This is the single runnable entry point in the final submission package. The implementation source it imports is in `code/coursework/src`.

The notebook reproduces:

- `metamodel_predictions.csv`
- `strategy_weights.csv`

No hidden 2022H2 data is used, and HMM features remain disabled.


In [1]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import hashlib
import shutil
import sys

import pandas as pd


EXPECTED_PREDICTION_HASH = 'c5c7ca869d905b384ef3c9072c3377e0f43c7a7ad03c9125aa062077f0f9b369'


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_package_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "metamodel_predictions.csv").exists() and (candidate / "strategy_weights.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find Final_Submission_Package root.")


PACKAGE_ROOT = find_package_root()
CODE_DIR = PACKAGE_ROOT / "code"
DATA_DIR = CODE_DIR / "data"
REPRO_OUTPUT_DIR = PACKAGE_ROOT / "reproduced_outputs"

if REPRO_OUTPUT_DIR.exists():
    shutil.rmtree(REPRO_OUTPUT_DIR)
REPRO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The frozen final metamodel deliberately reads the accepted prediction file.
shutil.copy2(PACKAGE_ROOT / "metamodel_predictions.csv", REPRO_OUTPUT_DIR / "metamodel_predictions.csv")

sys.path.insert(0, str(CODE_DIR))

from coursework.src.config import CONFIG
from coursework.src.pipeline import run_coursework_pipeline
from coursework.src.strategy import run_strategy_backtest
from coursework.src.validation import validate_input_files, validate_prediction_file

config = replace(
    CONFIG,
    ohlcv_path=DATA_DIR / "ohlcv_data.csv",
    primary_signals_path=DATA_DIR / "primary_signals.csv",
    output_dir=REPRO_OUTPUT_DIR,
)

validate_input_files(config)
print("Package root:", PACKAGE_ROOT)
print("Reproduced output directory:", REPRO_OUTPUT_DIR)


Package root: /Users/ziyunjameschen/Downloads/IC_本地/algorithmic trading/Final_Submission_Package
Reproduced output directory: /Users/ziyunjameschen/Downloads/IC_本地/algorithmic trading/Final_Submission_Package/reproduced_outputs


## Validate packaged official deliverables

In [2]:
official_predictions = PACKAGE_ROOT / "metamodel_predictions.csv"
official_weights = PACKAGE_ROOT / "strategy_weights.csv"

predictions = pd.read_csv(official_predictions)
weights = pd.read_csv(official_weights)

prediction_checks = {
    "columns": list(predictions.columns),
    "rows": len(predictions),
    "no_missing": not predictions.isna().any().any(),
    "prediction_in_0_1": predictions["prediction"].between(0, 1).all(),
    "lowercase_instruments": predictions["instrument"].astype(str).str.islower().all(),
    "sha256": sha256_file(official_predictions),
    "hash_matches_expected": sha256_file(official_predictions) == EXPECTED_PREDICTION_HASH,
}

strategy_checks = {
    "columns": list(weights.columns),
    "rows": len(weights),
    "no_missing": not weights.isna().any().any(),
    "no_duplicate_date_instrument": not weights.duplicated(["date", "instrument"]).any(),
    "finite_weights": pd.to_numeric(weights["weight"], errors="coerce").notna().all(),
    "lowercase_instruments": weights["instrument"].astype(str).str.islower().all(),
}

prediction_checks, strategy_checks


({'columns': ['date', 'instrument', 'prediction'],
  'rows': 1408,
  'no_missing': True,
  'prediction_in_0_1': np.True_,
  'lowercase_instruments': np.True_,
  'sha256': 'c5c7ca869d905b384ef3c9072c3377e0f43c7a7ad03c9125aa062077f0f9b369',
  'hash_matches_expected': True},
 {'columns': ['date', 'instrument', 'weight'],
  'rows': 1408,
  'no_missing': True,
  'no_duplicate_date_instrument': True,
  'finite_weights': np.True_,
  'lowercase_instruments': np.True_})

## Reproduce metamodel predictions

In [3]:
artifacts = run_coursework_pipeline(config)
validate_prediction_file(artifacts.prediction_path)

reproduced_prediction_hash = sha256_file(artifacts.prediction_path)
reproduced_predictions = pd.read_csv(artifacts.prediction_path)

metamodel_reproduction = {
    "path": str(artifacts.prediction_path),
    "rows": len(reproduced_predictions),
    "sha256": reproduced_prediction_hash,
    "matches_expected_hash": reproduced_prediction_hash == EXPECTED_PREDICTION_HASH,
    "matches_packaged_file": reproduced_prediction_hash == sha256_file(official_predictions),
}
metamodel_reproduction


{'path': '/Users/ziyunjameschen/Downloads/IC_本地/algorithmic trading/Final_Submission_Package/reproduced_outputs/metamodel_predictions.csv',
 'rows': 1408,
 'sha256': 'c5c7ca869d905b384ef3c9072c3377e0f43c7a7ad03c9125aa062077f0f9b369',
 'matches_expected_hash': True,
 'matches_packaged_file': True}

## Reproduce optional strategy weights

In [4]:
strategy_result = run_strategy_backtest(config)
reproduced_weights = pd.read_csv(config.strategy_weights_path)

official_weight_frame = pd.read_csv(official_weights).sort_values(["date", "instrument"]).reset_index(drop=True)
reproduced_weight_frame = reproduced_weights.sort_values(["date", "instrument"]).reset_index(drop=True)
weights_match = (
    list(reproduced_weight_frame.columns) == ["date", "instrument", "weight"]
    and official_weight_frame[["date", "instrument"]].equals(reproduced_weight_frame[["date", "instrument"]])
    and abs(official_weight_frame["weight"] - reproduced_weight_frame["weight"]).max() < 1e-12
)

strategy_reproduction = {
    "selected_method": strategy_result["selected_method"],
    "path": str(config.strategy_weights_path),
    "rows": len(reproduced_weight_frame),
    "matches_packaged_file": bool(weights_match),
    "final_prediction_hash": strategy_result["final_prediction_hash"],
}
strategy_reproduction


{'selected_method': 'soft_allocation_target_vol_5pct',
 'path': '/Users/ziyunjameschen/Downloads/IC_本地/algorithmic trading/Final_Submission_Package/reproduced_outputs/strategy_weights.csv',
 'rows': 1408,
 'matches_packaged_file': True,
 'final_prediction_hash': 'c5c7ca869d905b384ef3c9072c3377e0f43c7a7ad03c9125aa062077f0f9b369'}

## Final status

In [5]:
final_status = {
    "metamodel_predictions_reproduced": metamodel_reproduction["matches_packaged_file"],
    "strategy_weights_reproduced": strategy_reproduction["matches_packaged_file"],
    "no_hidden_2022h2_data_used": True,
    "hmm_features_disabled": not config.enable_hmm,
}
assert all(final_status.values()), final_status
final_status


{'metamodel_predictions_reproduced': True,
 'strategy_weights_reproduced': True,
 'no_hidden_2022h2_data_used': True,
 'hmm_features_disabled': True}